In [ ]:
# Kill all processes on the GPU
!fuser -v /dev/nvidia* -k

In [ ]:
# Check GPU status
!nvidia-smi

In [ ]:
# %%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    %pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    %pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install transformers==4.56.2
%pip install --no-deps trl==0.22.2

# Configuration

In [ ]:
# Fix TorchDynamo bug
import os
os.environ['TORCHDYNAMO_DISABLE'] = '1'
os.environ['UNSLOTH_DISABLE_COMPILE'] = '1'
os.environ['UNSLOTH_DISABLE_FUSED_KERNELS'] = '1'

In [ ]:
# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Data configuration
SFT_DATA_ID = 'jxie/coco_captions'
RL_DATA_ID = 'alxxtexxr/ViRL1.25K'
DATA_SPLIT = 'train'
SAVE_DATA_ID = 'alxxtexxr/BIVR-Data'

# # For small experiment
# DATA_SIZE = 100
# BATCH_SIZE = 10

# For larger experiment
DATA_SIZE = 1000
BATCH_SIZE = 100

# Model

In [ ]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
)

In [ ]:
from datasets import load_dataset, Dataset
from collections import deque

def load_hf_dataset(data_id, split, total_size, from_end=False):
    dataset_stream = load_dataset(data_id, split=split, streaming=True)
    unique_cocoids = set()

    if not from_end:
        # Slice from the start (original behavior)
        sliced_data = []
        for example in dataset_stream:
            if len(sliced_data) >= total_size:
                break

            cocoid = example.get('cocoid', None)
            if cocoid is not None:
                if cocoid in unique_cocoids:
                    continue
                unique_cocoids.add(cocoid)

            sliced_data.append(example)
    else:
        # Slice from the end using a fixed-size buffer
        buffer = deque(maxlen=total_size)

        for example in dataset_stream:
            cocoid = example.get('cocoid', None)
            if cocoid is not None:
                if cocoid in unique_cocoids:
                    continue
                unique_cocoids.add(cocoid)

            buffer.append(example)

        sliced_data = list(buffer)

    dataset = Dataset.from_list(sliced_data)
    return dataset

sft_dataset = load_hf_dataset(SFT_DATA_ID, split=DATA_SPLIT, total_size=DATA_SIZE, from_end=False)
rl_dataset = load_hf_dataset(RL_DATA_ID, split=DATA_SPLIT, total_size=DATA_SIZE, from_end=False)

print("SFT dataset:")
print(sft_dataset)
print()
print("RL dataset:")
print(rl_dataset)

In [ ]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

# if 'url' in dataset.features:
#     # Load and process images from URLs
#     dataset = dataset.map(
#         process_image_with_url, 
#         num_proc=4,
#     )
# else:
#     # Resize to (512, 512) and convert to RGB
#     dataset = dataset.map(
#         process_image, 
#         # num_proc=4, # Somehow multiprocessing causes errors on T4
#     )

sft_dataset = sft_dataset.map(
    process_image,
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)
rl_dataset = rl_dataset.map(
    process_image, 
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)

In [ ]:
from datasets import concatenate_datasets

# Keep only decoded_image + add source label
sft_images = sft_dataset.select_columns(['decoded_image']).map(lambda x: {'source': 'sft'})
rl_images = rl_dataset.select_columns(['decoded_image']).map(lambda x: {'source': 'rl'})

# Combine
train_dataset = concatenate_datasets([sft_images, rl_images])

# Rename column
train_dataset = train_dataset.rename_column('decoded_image', 'image')

print(train_dataset)

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

In [ ]:
import math
import os
from pathlib import Path
import numpy as np
from huggingface_hub import upload_file
from sklearn.preprocessing import StandardScaler

train_size = len(train_dataset)
save_dir = Path(
    f"model_{MODEL_ID.replace('/', '_')}"
    f".data_sft_{SFT_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
    f".data_rl_{RL_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
)
os.makedirs(save_dir, exist_ok=True)

scaler = StandardScaler()

for i in range(math.ceil(train_size / BATCH_SIZE)):
    start_idx = i * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)

    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    # batch = visual_embs.float() # Convert to float32 for statistics calculation -> (N, 2048)

    # # Statistics for Standardization
    # batch_n = batch.shape[0]
    # batch_sum = batch.sum(dim=0)
    # batch_sum_sq = (batch**2).sum(dim=0)

    # # Statistics for Min-Max Normalization
    # batch_min = batch.min(dim=0).values
    # batch_max = batch.max(dim=0).values

    # print("Batch size:", batch_n)
    # print("Batch sum average:", batch_sum.mean())
    # print("Batch sum squared average:", batch_sum_sq.mean())
    # print("Batch min average:", batch_min.mean())
    # print("Batch max average:", batch_max.mean())
    # print()

    # Partial fit the scaler with current batch
    scaler.partial_fit(visual_embs.detach().cpu().float().numpy())

    # Save vision data and statistics
    range_tag = f'{str(start_idx).zfill(len(str(DATA_SIZE)))}-{str(end_idx-1).zfill(len(str(DATA_SIZE)))}'
    vision_data_path = save_dir / f'vision_data_{range_tag}.npz'
    # stats_path = save_dir / f'{range_tag}.stats.npz'
    
    np.savez(
        vision_data_path,
        pixel_values=pixel_values.cpu().numpy(),
        image_grid_thw=image_grid_thw.cpu().numpy(),
        visual_embs=visual_embs.detach().cpu().float().numpy(),
    )
    # np.savez(
    #     stats_path,
    #     batch_n=batch_n,
    #     batch_sum=batch_sum.cpu().numpy(),
    #     batch_sum_sq=batch_sum_sq.cpu().numpy(),
    #     batch_min=batch_min.cpu().numpy(),
    #     batch_max=batch_max.cpu().numpy()
    # )

    print("Vision data saved to:", vision_data_path)
    # print("Statistics saved to:", stats_path)

    upload_file(
        path_or_fileobj=str(vision_data_path),
        path_in_repo=str(vision_data_path),
        repo_id=SAVE_DATA_ID,
        repo_type='dataset',
    )
    # upload_file(
    #     path_or_fileobj=str(stats_path),
    #     path_in_repo=str(stats_path),
    #     repo_id=SAVE_DATA_ID,
    #     repo_type='dataset',
    # )

In [ ]:
import joblib

# Save the scaler
scaler_path = save_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

upload_file(
    path_or_fileobj=str(scaler_path),
    path_in_repo=str(scaler_path),
    repo_id='alxxtexxr/BIVR-Data',
    repo_type='dataset',
)